[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NTU-CompHydroMet-Lab/Datacube-demo/blob/main/ODC/notebooks/01_odc_load_demo.ipynb)

# Open Data Cube   
The Open Data Cube (ODC) is an open source solution for accessing, managing, and analyzing large quantities of Geographic Information System (GIS) data - namely Earth Observation (EO) data. It presents a common analytical framework composed of a series of data structures and tools which facilitate the organization and analysis of large gridded data collections.

Open Data Cube（ODC）是一個開源的地球觀測資料管理與分析框架。它的核心想法是：
**與其管理一堆散落的影像檔案，不如把它們「索引」起來，變成一個可以用時間、空間、產品名稱直接查詢的資料立方（Data Cube）**。

使用者不需要知道檔案放在哪裡、檔名是什麼、座標系統為何——只要下一個查詢
（「給我某產品、某範圍、某段時間的資料」），ODC 就會自動找出相關檔案、裁切、
重投影、堆疊成帶時間維度的陣列（xarray）供分析使用。

### 本教學會做什麼

1. 在 Colab 中從零建立一個迷你 ODC 環境（PostgreSQL 索引資料庫 + `datacube` 套件）。
2. 匯入 4 個年度的台灣 Sentinel-2 土地覆蓋（land cover）GeoTIFF 並建立索引。
3. 用 `dc.load()` 查詢大台北地區的資料，做跨年度地覆面積統計與地圖視覺化。

跑完你會體會到 ODC 最核心的價值：**分析程式碼只描述「想要什麼資料」，
不處理「資料在哪、怎麼讀」**——同樣的程式碼面對 4 個檔案或 4 萬個檔案完全相同。

ODC 的系統架構與生態系可參考官方介紹：[Open Data Cube Overview](https://www.opendatacube.org/overview-draft)

## ODC Sentinel-2 Land-Cover Load Demo

This notebook loads a fixed Greater Taipei bounding box from the indexed Taiwan Sentinel-2-derived annual land-cover GeoTIFFs.

`x` and `y` are projected raster coordinates in EPSG:32651. The plot below relabels ticks as longitude and latitude for readability. The land-cover class codes are stored in the `classification` data variable. NoData is class `0`.

### 關於這份示範資料

- **內容**：由 Sentinel-2 衛星影像衍生的台灣**年度土地覆蓋分類圖**，共 4 個年度（2017–2020）。
- **座標系統**：EPSG:32651（UTM Zone 51N，單位為公尺），解析度 10 公尺。
- **像元值**：土地覆蓋類別碼（1 水體、2 樹林、5 農作、7 建成區……），`0` 代表無資料（NoData）。

後面圖表的 `x`、`y` 軸是投影座標，繪圖時會把刻度轉回經緯度方便閱讀。

## Colab Setup

This section prepares a self-contained ODC environment inside the Colab VM:

1. Install and start PostgreSQL, then create the `datacube` database and user.
2. Install the `datacube` Python package (same version as the local Docker demo).
3. Clone this repository — Git LFS pulls the demo GeoTIFFs (~370 MB, takes a few minutes).
4. Initialize the ODC schema, add the `s2_landcover_taiwan` product, and index the demo datasets.

The full setup takes roughly 3–5 minutes on a fresh Colab runtime.

All setup cells are guarded by `IN_COLAB`, so they are safe no-ops when this notebook
runs in the local Docker environment (`odc_local_demo`), where the index is already
built by `setup_odc_demo.sh`.

### 中文說明

一個 ODC 環境由三個部分組成：**PostgreSQL 資料庫**（存放索引與 metadata）、
**`datacube` Python 套件**（查詢與載入介面）、以及**實體資料檔**（GeoTIFF）。
正式環境中這些由伺服器統一管理、只需建置一次；在 Colab 上我們則是每次開新的
runtime 都現場搭一套迷你版，大約需要 3–5 分鐘。步驟對應如下：

| 步驟 | 動作 | 對應正式環境 |
| --- | --- | --- |
| 1 | 安裝並啟動 PostgreSQL、建立資料庫 | DB 伺服器（常駐） |
| 2 | 安裝 `datacube` 套件 | 分析環境 |
| 3 | 下載示範 GeoTIFF（Git LFS，約 370 MB） | 資料儲存區 |
| 4 | 初始化 schema、註冊產品、建立索引 | 資料匯入流程（只做一次） |

### Step 0：偵測執行環境

先判斷目前是否在 Colab 上。後面的安裝格都以 `if IN_COLAB:` 保護——
如果你是在本地 Docker 環境（`odc_local_demo`）開這本 notebook，這些格子會直接跳過，
因為環境早已由 `setup_odc_demo.sh` 建好。

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

### Step 1：安裝系統依賴

ODC 的「索引」存在 PostgreSQL 裡（記錄每個檔案的時空範圍、座標系統、路徑等 metadata），
所以第一步是把資料庫裝起來、開一個名為 `datacube` 的資料庫與使用者。
`git-lfs` 則是等一下下載示範 GeoTIFF 需要的工具。最後安裝 `datacube` 套件
（與本 repo Docker 示範環境相同版本，確保行為一致）。

> 安裝過程若出現 pip 相依性警告（例如 SQLAlchemy 降版）屬正常現象，不影響後續執行。

In [ ]:
if IN_COLAB:
    # PostgreSQL backs the ODC index; git-lfs is needed to pull the demo GeoTIFFs.
    !sudo apt-get -qq update
    !sudo apt-get -qq install -y postgresql git-lfs
    !sudo service postgresql start
    !sudo -u postgres psql -tc "SELECT 1 FROM pg_roles WHERE rolname='datacube'" | grep -q 1 || sudo -u postgres psql -c "CREATE USER datacube WITH PASSWORD 'datacube';"
    !sudo -u postgres psql -tc "SELECT 1 FROM pg_database WHERE datname='datacube'" | grep -q 1 || sudo -u postgres createdb -O datacube datacube
    %pip install -q datacube==1.8.19 psycopg2-binary==2.9.9 pyyaml

### Step 2：下載資料並建立索引

這一格做的事就是資料匯入 ODC 的標準流程，先認識兩個核心概念：

- **Product（產品）**：一「類」資料的定義——名稱、波段（measurement）、資料型別、NoData 值等。
  這裡註冊的產品是 `s2_landcover_taiwan`。
- **Dataset（資料集）**：產品底下的一筆實體資料——通常是一個時間切片對應的檔案。
  4 個年度的 GeoTIFF 會產生 4 個 datasets。

流程：`datacube system init` 建立資料庫 schema → `datacube product add` 註冊產品定義
→ 為每個 GeoTIFF 產生描述其時空範圍的 YAML → `datacube dataset add` 寫入索引。
**索引只記 metadata 與檔案路徑，不複製資料本身。**

> 技術細節：`system init` 需要較高的資料庫權限（會建立 DB roles），
> 因此以 `postgres` 超級使用者執行，完成後再把 `agdc` schema 的權限授予 `datacube` 使用者。

In [ ]:
if IN_COLAB:
    import os

    REPO_DIR = "/content/Datacube-demo"
    DEMO_ROOT = f"{REPO_DIR}/ODC/odc_local_demo"

    if not os.path.exists(REPO_DIR):
        !git lfs install --skip-repo
        !git clone https://github.com/NTU-CompHydroMet-Lab/Datacube-demo.git {REPO_DIR}

    # Same connection settings that setup_odc_demo.sh writes inside the Docker container.
    # Write the config to a known, universally accessible location like /tmp.
    DATACUBE_CONFIG_FILE = "/tmp/.datacube.conf"
    with open(DATACUBE_CONFIG_FILE, "w") as f:
        f.write(
            "[datacube]\n"
            "db_hostname: localhost\n"
            "db_database: datacube\n"
            "db_username: datacube\n"
            "db_password: datacube\n"
        )

    # Set the DATACUBE_CONFIG_PATH environment variable for Python processes
    # and for the shell commands that follow.
    os.environ['DATACUBE_CONFIG_PATH'] = DATACUBE_CONFIG_FILE

    # Execute datacube commands as the postgres system user.
    # Pass DATACUBE_CONFIG_PATH as an environment variable to each command executed by sudo.

    # Run datacube system init as the postgres system user,
    # and override the DB_USERNAME environment variable to 'postgres'
    # so it connects with superuser privileges to create the schema.
    !sudo -u postgres DATACUBE_CONFIG_PATH={DATACUBE_CONFIG_FILE} DB_USERNAME=postgres datacube system init

    # Grant privileges to the 'datacube' user on the 'agdc' schema and its objects.
    # This ensures the 'datacube' user can access objects created by 'postgres' user during init.
    !sudo -u postgres psql -d datacube -c "GRANT USAGE ON SCHEMA agdc TO datacube;"
    !sudo -u postgres psql -d datacube -c "GRANT ALL PRIVILEGES ON ALL TABLES IN SCHEMA agdc TO datacube;"
    !sudo -u postgres psql -d datacube -c "ALTER DEFAULT PRIVILEGES IN SCHEMA agdc GRANT ALL PRIVILEGES ON TABLES TO datacube;"
    !sudo -u postgres psql -d datacube -c "GRANT ALL PRIVILEGES ON ALL SEQUENCES IN SCHEMA agdc TO datacube;"
    !sudo -u postgres psql -d datacube -c "ALTER DEFAULT PRIVILEGES IN SCHEMA agdc GRANT ALL PRIVILEGES ON SEQUENCES TO datacube;"

    # Verify system setup before proceeding.
    # Other commands can connect as the 'datacube' DB user, so no DB_USERNAME override is needed.
    !sudo -u postgres DATACUBE_CONFIG_PATH={DATACUBE_CONFIG_FILE} datacube system check

    !sudo -u postgres DATACUBE_CONFIG_PATH={DATACUBE_CONFIG_FILE} datacube product add {DEMO_ROOT}/products/s2_landcover_taiwan.yaml
    # The python script itself doesn't need the datacube config path, as it's not calling datacube CLI directly.
    !python {DEMO_ROOT}/scripts/write_dataset_yaml.py --data-dir {DEMO_ROOT}/data --dataset-dir {DEMO_ROOT}/datasets --product s2_landcover_taiwan --measurement classification
    !sudo -u postgres DATACUBE_CONFIG_PATH={DATACUBE_CONFIG_FILE} datacube dataset add {DEMO_ROOT}/datasets/*.yaml

### Step 3：驗證環境

用 repo 內建的檢查腳本確認：產品已註冊、4 個 datasets 已入索引、抽樣載入成功。
看到 `OK` 字樣即代表資料立方已就緒。

In [ ]:
if IN_COLAB:
    # Sanity check: product indexed, datasets found, sample load succeeds.
    !DATACUBE_CONFIG_PATH={DATACUBE_CONFIG_FILE} python {DEMO_ROOT}/scripts/check_odc_demo.py

## 連線資料立方

環境就緒後，分析端的入口只有一個：`Datacube` 物件。它透過設定檔（或環境變數）
找到 PostgreSQL 索引，之後所有查詢、載入都經由它。
**注意：從這裡開始的程式碼與正式環境完全相同**——不管背後是 Colab 的迷你環境、
本地 Docker，還是伺服器上的正式部署。

In [ ]:
from datacube import Datacube
from rasterio.warp import transform, transform_bounds
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
import matplotlib.patches as mpatches

PRODUCT = "s2_landcover_taiwan"
MEASUREMENT = "classification"
OUTPUT_CRS = "EPSG:32651"
RESOLUTION = (-10, 10)

# On Colab, Datacube() picks up the DATACUBE_CONFIG_PATH env var set in the setup cell.
dc = Datacube(app="01_odc_load_demo")

### 查詢有哪些產品

`dc.list_products()` 列出索引中註冊的所有產品及其定義。
在正式環境中，這就是使用者探索「這個資料立方裡有什麼資料可用」的起點。

In [ ]:
dc.list_products().loc[[PRODUCT]]

### 查詢產品底下的資料集

`dc.index.datasets.search()` 回傳該產品底下所有已索引的 datasets。
這裡應該看到 **4 筆**——對應 2017–2020 四個年度的土地覆蓋圖。
每筆記錄包含時間範圍、空間範圍與實體檔案位置。

In [ ]:
datasets = list(dc.index.datasets.search(product=PRODUCT))
len(datasets), datasets[:1]

## Taipei Bounding Box

The bbox below is a focused Taipei core area in lon/lat. It is transformed to EPSG:32651 before loading from ODC.

### 中文說明

我們以經緯度（EPSG:4326）定義大台北核心區的範圍框，再用 `transform_bounds`
轉換成資料本身的投影座標系（EPSG:32651）。這是使用投影資料時的常見步驟：
**人習慣用經緯度描述範圍，資料以公尺網格儲存**，查詢前先把範圍換算過去。

In [ ]:
# Greater Taipei core bbox: west, south, east, north in EPSG:4326
# Use a focused bbox for an interactive notebook; expand only when needed.
taipei_bbox_lonlat = (121.40, 24.95, 121.70, 25.20)

min_x, min_y, max_x, max_y = transform_bounds(
    "EPSG:4326",
    OUTPUT_CRS,
    *taipei_bbox_lonlat,
    densify_pts=21,
)

taipei_query = {
    "x": (min_x, max_x),
    "y": (min_y, max_y),
}
taipei_query

## 核心步驟：`dc.load()` 載入資料

這一行是整個 ODC 工作流程的核心。你只描述**想要什麼**：哪個產品、哪個範圍、
什麼座標系統、什麼解析度——ODC 自動完成以下所有工作：

1. 查詢索引，找出與範圍相交的所有檔案；
2. 讀取每個檔案中**需要的部分**（不是整檔載入）；
3. 裁切、對齊到指定網格；
4. 把不同時間的資料**堆疊成帶 `time` 維度的 xarray Dataset**。

對照傳統做法——自己開 4 個檔案、逐一裁切、對齊網格、手動堆疊——差異已經很明顯；
而當正式環境中同一產品底下有數千個影像切片時，**這行程式碼一個字都不用改**，
索引會幫你篩掉不相關的檔案。這就是資料立方架構的價值所在。

In [ ]:
taipei = dc.load(
    product=PRODUCT,
    measurements=[MEASUREMENT],
    x=taipei_query["x"],
    y=taipei_query["y"],
    crs=OUTPUT_CRS,
    output_crs=OUTPUT_CRS,
    resolution=RESOLUTION,
)
taipei

### 認識載入結果：xarray

`dc.load()` 回傳的是 `xarray.Dataset`，取出 `classification` 這個變數後得到
`xarray.DataArray`，維度為 `(time, y, x)`：4 個年度 × 空間網格。
xarray 讓你能以座標（而非陣列索引）選取資料，例如 `sel(time=...)` 取某個年度，
是 Python 地球科學生態系的標準資料結構。

In [ ]:
classification = taipei[MEASUREMENT]
classification

## Area Summary

This avoids converting the full raster to a pandas Series. It counts each class directly with NumPy, then creates a small summary table.

### 中文說明

有了帶時間維度的陣列，跨年度分析就是簡單的陣列運算：逐年統計每個地覆類別的
像元數，乘上單一像元面積（10 m × 10 m = 100 m²）換算成平方公里。
從結果表可以直接觀察大台北地區各年度建成區、樹林、水體等面積的變化。

In [ ]:
class_names = {
    1: "Water",
    2: "Trees",
    4: "Flooded Vegetation",
    5: "Crops",
    7: "Built Area",
    8: "Bare Ground",
    9: "Snow/Ice",
    10: "Clouds",
    11: "Rangeland",
}

pixel_area_m2 = 100
class_codes = list(class_names)

rows = []
for time_value in classification.time.values:
    arr = classification.sel(time=time_value).values
    for code in class_codes:
        pixel_count = int(np.count_nonzero(arr == code))
        rows.append(
            {
                "time": str(time_value)[:10],
                "class_code": code,
                "class_name": class_names[code],
                "pixel_count": pixel_count,
                "area_km2": pixel_count * pixel_area_m2 / 1_000_000,
            }
        )

area = pd.DataFrame(rows)
area

## Plot

The plot uses one time slice and decimates pixels for display only. The raster is still loaded in EPSG:32651, but axis tick labels are converted back to longitude and latitude.

### 中文說明

最後把某一年度的分類結果畫成地圖：以官方配色為每個地覆類別上色、
NoData 設為透明、座標刻度轉回經緯度。`display_step` 只是縮小繪圖用的取樣間隔，
不影響前面統計的完整解析度。

In [ ]:
class_colors = {
    1: "#419BDF",
    2: "#397D49",
    4: "#7A87C6",
    5: "#E49635",
    7: "#C4281B",
    8: "#A59B8F",
    9: "#B39FE1",
    10: "#FFFFFF",
    11: "#E3E2C3",
}

codes = list(class_colors)
bounds = [0.5, 1.5, 2.5, 4.5, 5.5, 7.5, 8.5, 9.5, 10.5, 11.5]
cmap = ListedColormap([class_colors[code] for code in codes])
norm = BoundaryNorm(bounds, cmap.N)


def set_lonlat_ticks(ax, x_values, y_values, x_count=5, y_count=5):
    x_ticks = np.linspace(float(x_values.min()), float(x_values.max()), x_count)
    y_ticks = np.linspace(float(y_values.min()), float(y_values.max()), y_count)
    center_x = float(x_values.mean())
    center_y = float(y_values.mean())

    lon_labels, _ = transform(OUTPUT_CRS, "EPSG:4326", x_ticks.tolist(), [center_y] * len(x_ticks))
    _, lat_labels = transform(OUTPUT_CRS, "EPSG:4326", [center_x] * len(y_ticks), y_ticks.tolist())

    ax.set_xticks(x_ticks)
    ax.set_yticks(y_ticks)
    ax.set_xticklabels([f"{lon:.2f}" for lon in lon_labels])
    ax.set_yticklabels([f"{lat:.2f}" for lat in lat_labels])


plot_time_index = 0
display_step = 10

plot_data = classification.isel(time=plot_time_index)
plot_data = plot_data.where(plot_data != 0)
plot_data = plot_data.isel(y=slice(None, None, display_step), x=slice(None, None, display_step))

fig, ax = plt.subplots(figsize=(10, 9))
plot_data.plot.imshow(ax=ax, cmap=cmap, norm=norm, add_colorbar=False)
set_lonlat_ticks(ax, plot_data.x, plot_data.y)

time_value = str(classification.time.values[plot_time_index])[:10]
ax.set_title(f"Greater Taipei Land Cover - {time_value}")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

patches = [
    mpatches.Patch(color=class_colors[code], label=f"{code} {class_names[code]}")
    for code in codes
]
ax.legend(handles=patches, loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
plt.tight_layout()
plt.show()

## 小結

本教學完整走過一個資料立方的生命週期：

1. **建置**：PostgreSQL + `datacube` 套件 + 資料檔；
2. **匯入**：註冊產品定義、為資料建立索引（只記 metadata，不搬資料）；
3. **使用**：`Datacube()` 連線 → 查詢產品/資料集 → `dc.load()` 取得分析就緒的 xarray → 統計與視覺化。

Colab 環境是拋棄式的（runtime 回收後索引即消失），適合教學體驗；
正式部署請參考 repo 內的 [`odc_local_demo`](https://github.com/NTU-CompHydroMet-Lab/Datacube-demo/tree/main/ODC/odc_local_demo)
（Docker + 常駐 PostgreSQL，索引建一次即可持續使用）。

延伸練習：

- 修改 `taipei_bbox_lonlat` 查詢其他地區（資料涵蓋全台）；
- 依 `odc_local_demo` README 的檔名規則放入自己的 GeoTIFF，建立自己的產品與索引。